[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/26.%20The%20Gate%20Automation%20%26%20Damage%20Detection%20Problem/P26-Tier-1-CLEAN_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# Problem 26: The Gate Automation & Damage Detection Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Learning Objectives

After completing this tier, you will be able to:
- Formulate gate automation as a mixed-integer programming problem
- Model sensor monitoring and failure detection mathematically
- Understand constraint satisfaction for maintenance scheduling
- Apply optimization to minimize downtime costs

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Gate Automation Mathematical Formulation')

Gate Automation Mathematical Formulation


### Data Structures and Example

Create the data structures for gates and sensors based on the source material.

In [ ]:
@dataclass
class Gate:
    id: int
    location: str
    vehicles_per_hour: int
    criticality: float
    downtime_cost_per_hour: float

@dataclass
class Sensor:
    id: int
    gate_id: int
    sensor_type: str
    monitoring_cost: float
    failure_rate: float
    criticality: float

# Create example data from source
gates = [
    Gate(id=1, location='Main_Entrance', vehicles_per_hour=150, criticality=0.95, downtime_cost_per_hour=15000),
    Gate(id=2, location='Exit_Gate', vehicles_per_hour=100, criticality=0.85, downtime_cost_per_hour=10000),
]

sensors = [
    Sensor(id=1, gate_id=1, sensor_type='photo_eye', monitoring_cost=5, failure_rate=0.02, criticality=0.9),
    Sensor(id=2, gate_id=1, sensor_type='ground_loop', monitoring_cost=9, failure_rate=0.03, criticality=0.7),
    Sensor(id=3, gate_id=1, sensor_type='safety_beam', monitoring_cost=7, failure_rate=0.015, criticality=1.0),
    Sensor(id=4, gate_id=2, sensor_type='photo_eye', monitoring_cost=5, failure_rate=0.025, criticality=0.8),
    Sensor(id=5, gate_id=2, sensor_type='ground_loop', monitoring_cost=9, failure_rate=0.035, criticality=0.6),
]

print(f'Gates: {len(gates)}')
print(f'Sensors: {len(sensors)}')
print(f'Downtime costs: Gate 1=${gates[0].downtime_cost_per_hour}, Gate 2=${gates[1].downtime_cost_per_hour}')

Gates: 2
Sensors: 5
Downtime costs: Gate 1=$15000, Gate 2=$10000


### Mathematical Model

Decision variables:
- x_{gst}: 1 if sensor s at gate g is monitored in period t
- y_{gtf}: 1 if failure mode f occurs at gate g in period t
- z_{gt}: 1 if gate g is operational in period t

Objective: Minimize sum(monitoring_costs) + sum(downtime_costs)

In [ ]:
# Simplified optimization approach
class GateAutomationModel:
    def __init__(self, gates, sensors):
        self.gates = gates
        self.sensors = sensors
        self.monitoring_decisions = {}
        self.sensor_health = {}
        self.gate_operational = {}
        
    def optimize_schedule(self):
        # Simple greedy approach for demonstration
        for hour in range(24):
            # Monitor critical sensors first
            for sensor in sorted(self.sensors, key=lambda x: x.criticality, reverse=True)[:2]:
                self.monitoring_decisions[(sensor.gate_id, sensor.id, hour)] = 1
                
                # Update sensor health
                if hour == 0:
                    self.sensor_health[(sensor.gate_id, sensor.id, hour)] = 1.0
                else:
                    prev_health = self.sensor_health.get((sensor.gate_id, sensor.id, hour-1), 1.0)
                    new_health = min(1.0, prev_health + 0.05)  # Monitoring improves health
                    self.sensor_health[(sensor.gate_id, sensor.id, hour)] = new_health
            
            # Update gate operational status
            for gate in self.gates:
                gate_sensors = [s for s in self.sensors if s.gate_id == gate.id]
                avg_health = np.mean([self.sensor_health.get((gate.id, s.id, hour), 1.0) for s in gate_sensors])
                self.gate_operational[(gate.id, hour)] = 1 if avg_health > 0.2 else 0
    
    def calculate_costs(self):
        monitoring_cost = 0
        downtime_cost = 0
        
        for sensor in self.sensors:
            for hour in range(24):
                if self.monitoring_decisions.get((sensor.gate_id, sensor.id, hour), 0) == 1:
                    monitoring_cost += sensor.monitoring_cost
        
        for gate in self.gates:
            for hour in range(24):
                if self.gate_operational.get((gate.id, hour), 1) == 0:
                    downtime_cost += gate.downtime_cost_per_hour * 0.1  # Normalized
        
        return monitoring_cost, downtime_cost

# Run optimization
model = GateAutomationModel(gates, sensors)
model.optimize_schedule()
monitoring_cost, downtime_cost = model.calculate_costs()
total_cost = monitoring_cost + downtime_cost

print(f'Monitoring cost: ${monitoring_cost:,.2f}')
print(f'Downtime cost: ${downtime_cost:,.2f}')
print(f'Total cost: ${total_cost:,.2f}')
print(f'System availability: {np.mean([model.gate_operational.get((g.id, h), 1) for g in gates for h in range(24)]) * 100:.1f}%')

Monitoring cost: $288.00
Downtime cost: $0.00
Total cost: $288.00
System availability: 100.0%


### Visualization

Create visualizations to understand the optimization results.

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Gate Automation Mathematical Formulation', fontsize=14, fontweight='bold')

# 1. Monitoring schedule
ax1 = axes[0, 0]
monitoring_matrix = np.zeros((len(gates), 24))
for gate in gates:
    for hour in range(24):
        monitored_count = sum(1 for s in sensors 
                           if s.gate_id == gate.id and model.monitoring_decisions.get((gate.id, s.id, hour), 0) == 1)
        monitoring_matrix[gate.id-1, hour] = monitored_count

ax1.imshow(monitoring_matrix, cmap='RdYlGn', aspect='auto')
ax1.set_title('Sensor Monitoring Schedule')
ax1.set_xlabel('Hour')
ax1.set_ylabel('Gate')

# 2. Sensor health evolution
ax2 = axes[0, 1]
for sensor in sensors[:3]:  # Show first 3 sensors
    health_values = [model.sensor_health.get((sensor.gate_id, sensor.id, h), 1.0) for h in range(24)]
    ax2.plot(range(24), health_values, label=f'G{sensor.gate_id}-{sensor.sensor_type}', marker='o', markersize=3)
ax2.set_title('Sensor Health Evolution')
ax2.set_xlabel('Hour')
ax2.set_ylabel('Health Index')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Gate operational status
ax3 = axes[1, 0]
for gate in gates:
    operational = [model.gate_operational.get((gate.id, h), 1) for h in range(24)]
    ax3.plot(range(24), operational, label=f'Gate {gate.id}', marker='s', markersize=4)
ax3.set_title('Gate Operational Status')
ax3.set_xlabel('Hour')
ax3.set_ylabel('Operational (1=Yes, 0=No)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Cost breakdown
ax4 = axes[1, 1]
costs = ['Monitoring', 'Downtime', 'Total']
values = [monitoring_cost, downtime_cost, total_cost]
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax4.bar(costs, values, color=colors)
ax4.set_title('Cost Breakdown')
ax4.set_ylabel('Cost ($)')
ax4.grid(axis='y', alpha=0.3)

for bar, value in zip(bars, values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.02,
            f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print('Mathematical formulation visualization completed!')

Mathematical formulation visualization completed!


### Why This Tier Exists

**Mathematical Formulation Advantages:**
- Provides provably optimal solutions
- Captures all constraints explicitly
- Serves as benchmark for other methods
- Enables systematic sensitivity analysis

**Limitations:**
- Computationally expensive for large instances
- Requires complete parameter information
- Limited flexibility for real-time decisions

**When to Use:**
- Small to medium facilities where optimality is critical
- Strategic planning and capacity design
- Algorithm benchmark development
- Situations with ample computational resources